In [1]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

/Users/ayhan/Desktop/DS_Bootcamp/Capstone_Martin_Ayhan/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### Wir bauen nun einen zweiten Embedding Schritt. Der erste Embedding Schritt war für die Issue_Detection. Ziel war es dabei Clustering zu machen. Der zweite Schritt des Embeddings ist für Semantic Search, RAG und AI Insights.

In [2]:
#Reviews laden
import pandas as pd

df=pd.read_csv("../data/BMW/clean_reviews.csv")
texts = df["content"].astype(str).tolist()
print("Number of reviews:", len(texts))

Number of reviews: 10509


In [3]:
#Embeddings erstellen
embeddings = model.encode(texts, batch_size=32, show_progress_bar = True)
print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 329/329 [00:33<00:00,  9.85it/s]

Embeddings shape: (10509, 384)


In [4]:
#FAISS Index bauen
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))
print("Total vectors in index:", index.ntotal)

Total vectors in index: 10509


In [5]:
#Semantic Search testen
query = "BMW app crashes when opening vehicle status"
query_embedding = model.encode([query])
distances, indices = index.search(query_embedding, k=5)
df.iloc[indices[0]][["content"]]

,content
329,"since last update, app crashes at login with n..."
9753,app kann nicht mehr auf fahrzeugstatus zugreif...
3595,my car disappeared from the app. tried adding ...
85,at 1st i thought that there is a problem with ...
3330,app is not working properly. vehicle finder an...


In [6]:
#Index speichern
faiss.write_index(index, "../models/review_index.faiss")

In [7]:
#Embeddings speichern
np.save("../models/review_embeddings.npy", embeddings)

In [ ]:
#------------------------------------------------------------------------------

In [1]:
#Imports
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/ayhan/Desktop/DS_Bootcamp/Capstone_Martin_Ayhan/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Daten laden
df = pd.read_csv("../data/BMW/reviews_with_clusters.csv")

print("Shape:", df.shape)
df.head()

Shape: (4035, 6)


,text,rating,date,sentiment,cluster,cluster_label
0,I have a vehicle with comfort access and a Sam...,2,2026-03-07 14:42:54,negative,0,"Based on the text, I would choose the label:\n..."
1,Can't add my e34 and e39 :/,1,2026-03-07 09:50:37,negative,2,"Based on your request, I've analyzed the issue..."
2,The lack of support for Octopus Intelligent Go...,2,2026-03-06 19:38:16,negative,0,"Based on the text, I would choose the label:\n..."
3,To have my BMW charge overnight in a 'time slo...,2,2026-03-05 08:46:07,negative,0,"Based on the text, I would choose the label:\n..."
4,Loss of push notifications issue is now fixed ...,2,2026-03-04 01:13:00,negative,2,"Based on your request, I've analyzed the issue..."


In [3]:
#Sicherheitscheck
assert "text" in df.columns, "❌ 'text' column missing"


In [4]:
#Modell laden
model = SentenceTransformer("all-MiniLM-L6-v2")


In [5]:
#Embeddings erzeugen
texts = df["text"].astype(str).tolist()

embeddings = model.encode(
    texts,
    show_progress_bar=True,
    batch_size=32
)

Batches: 100%|██████████| 127/127 [00:16<00:00,  7.93it/s]


In [6]:
#Shape prüfen
print("Embeddings shape:", embeddings.shape)


Embeddings shape: (4035, 384)


In [7]:
#Speichern (Wichtig für spätere RAG nutzung)
np.save("../models/review_embeddings.npy", embeddings)


In [8]:
#DataFrame speichern
df.to_csv("../data/BMW/reviews_with_embeddings.csv", index=False)


In [9]:
#Quick Test (sehr wichtig -- Funktion zum finden ähnlicher Reviews)
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_reviews(query, top_k=5):
    
    query_embedding = model.encode([query])
    
    similarities = cosine_similarity(query_embedding, embeddings)[0]
    
    top_indices = similarities.argsort()[-top_k:][::-1]
    
    return df.iloc[top_indices][["text"]]

In [10]:
#Testen
find_similar_reviews("app keeps disconnecting")


,text
1035,keeps dropping out and won't reconnect
1310,Causes connection problem on wireless Andriod ...
789,Keeps logging out. What's the point in the app...
921,The app keeps log out of itself since the late...
1702,"Fails to open, Connected app works fine"
